# WERC 2026 · Redes multinivel y competencia tecnológica

**Dr. Sergio Gevatschnaider · Universidad de Palermo, Facultad de Ingeniería**

Este cuaderno acompaña la ponencia *Coordinación y competencia en la frontera de inteligencia artificial: un modelo de redes multinivel*. **No es el generador original de las 480 redes reportadas**. Ejecuta nuevas simulaciones didácticas reproducibles, además de verificar resultados algebraicos del manuscrito.

**Criterio de éxito:** reproducir las identidades exactas (softmax, estrella/camino, precio sombra, coordinación) y observar la sensibilidad de los escenarios sintéticos sin extrapolar a empresas reales.


## 0. Preparar bibliotecas

En Google Colab instalar el repositorio público. En un clon local con el paquete ya instalado, omitir la instalación.


In [ ]:
%pip -q install git+https://github.com/sgevatschnaider/BigData-Graphs-Evo-CA-Classroom.git


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from classroom.graphs.werc_2026 import softmax, nash_equilibrium, resource_equilibrium, cooperation_margin, make_network
np.set_printoptions(precision=6, suppress=True)


## 1. Reina Roja y softmax

Si todas las capacidades cambian por el mismo g, las cuotas permanecen constantes con beta fijo y el mismo conjunto de competidores. Esta identidad no demuestra ineficiencia económica.


In [ ]:
q=np.array([80.,80.]);beta=.1;g=10
before=softmax(q,beta);after=softmax(q+g,beta);unilateral=softmax(q+np.array([10,0]),beta)
print('Antes:',before,'\nProgreso común:',after,'\nMejora unilateral A:',unilateral)
assert np.allclose(before,after)
assert np.isclose(unilateral[0],1/(1+np.exp(-1)))


## 2. El juego reducido y la solución exacta de cuatro nodos

La fórmula interior se utiliza solamente cuando W>=0, b>0, v>0, alpha>=0 y rho(alpha D_b^-1 W)<1.


In [ ]:
Wstar=np.zeros((4,4));Wpath=np.zeros((4,4))
for j in [1,2,3]:Wstar[0,j]=Wstar[j,0]=1
for i in range(3):Wpath[i,i+1]=Wpath[i+1,i]=1
star,r1=nash_equilibrium(Wstar,alpha=.2)
path,r2=nash_equilibrium(Wpath,alpha=.2)
print('Estrella:',star,'total',star.sum(),'radio ajustado',r1)
print('Camino: ',path,'total',path.sum(),'radio ajustado',r2)
assert np.allclose(star,[20/11,15/11,15/11,15/11])
assert np.allclose(path,[25/19,30/19,30/19,25/19])


## 3. Capacidad escasa y precio sombra

Con un solo recurso homogéneo de oferta K y restricción vinculante: x*=(K/Z)z y mu=1-K/Z.


In [ ]:
x,mu,bound=resource_equilibrium(Wstar,capacity=4,eta=.2)
print('Esfuerzo restringido:',x,'\nPrecio sombra:',mu,'\nVinculante:',bound)
assert np.allclose(x,[16/13,12/13,12/13,12/13])
assert np.isclose(mu,21/65) and np.isclose(x.sum(),4)


## 4. Coordinación y descuento intertemporal

La desigualdad de no desviación presupone monitoreo, sanción y castigo futuro creíble. No conecta todavía H con detección estatal.


In [ ]:
G=.8;d=.5;P=.25;future=.6
delta_star=(G-d*P)/(G-d*P+d*future)
print('delta mínimo',delta_star)
assert np.isclose(delta_star,9/13)
assert np.isclose(cooperation_margin(G,d,P,delta_star,future),0)


## 5. Cuatro familias de redes sintéticas nuevas

Se generan grafos de 24 nodos y 48 aristas para ER, WS, BA y SBM. Los ajustes a m=48 afectan las distribuciones. **No** son las realizaciones del CSV WERC. Para comparar exactamente las 480 del trabajo necesitaríamos incorporar y auditar las filas originales y el código generador.


In [ ]:
families=['ER','WS','BA','SBM'];rows=[]
for fam in families:
    for replicate in range(120):
        W=make_network(fam,24,48,seed=20261020+replicate)
        eig=np.linalg.eigvalsh(W)
        rho=max(abs(eig))
        x,radius=nash_equilibrium(W,alpha=.12)
        rows.append({'familia':fam,'réplica':replicate,'rho':rho,'total_libre':x.sum(),'mu_homogenea':max(0.,1-36.923076923/x.sum())})
new_runs=pd.DataFrame(rows)
assert len(new_runs)==480
assert all(int(make_network(fam,24,48).sum()/2)==48 for fam in families)
print(new_runs.groupby('familia').agg(rho_media=('rho','mean'), esfuerzo_medio=('total_libre','mean'), esfuerzo_sd=('total_libre','std'), mu_media=('mu_homogenea','mean')).round(3))


### Control de visualización

El gráfico siguiente corresponde a **nuevas redes didácticas**, no a una reproducción numérica de las cifras del manuscrito. Que difieran puede deberse a elecciones del generador y sus ajustes.


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
new_runs.boxplot(column='total_libre',by='familia',ax=ax)
ax.set(title='Nuevas redes didácticas: inversión de equilibrio',xlabel='Familia',ylabel='Suma de esfuerzos');fig.suptitle('')
plt.tight_layout();plt.show()


## 6. Una extensión conceptual: evaluadores que cambian por época

**Advertencia**: se ilustra un criterio móvil; esto NO implementa Red Queen Gödel Machine ni sus algoritmos de promoción, anclas o borrado selectivo. El RQGM original es otra línea de investigación.


In [ ]:
epochs=np.arange(1,11)
agent_quality=1-.7*np.exp(-.14*epochs)
evaluator_static=np.full(len(epochs),.3)
evaluator_dynamic=.3+.035*(epochs-1)
score_static=agent_quality-evaluator_static
score_dynamic=agent_quality-evaluator_dynamic
plt.plot(epochs,score_static,label='Evaluador fijo (hipotético)')
plt.plot(epochs,score_dynamic,label='Criterio creciente (hipotético)')
plt.xlabel('Época');plt.ylabel('Puntuación pedagógica');plt.legend();plt.grid(alpha=.2);plt.show()


## 7. Registro de decisión y límites

**Decisión:** usar las fórmulas del juego reducido para comparar escenarios **sintéticos**, y mantener explícitas las hipótesis, el estado del experimento y sus datos de procedencia.

**Límites:** (a) solo se ha derivado el equilibrio reducido; (b) el CSV original WERC no está incorporado; (c) una identidad de cuota no demuestra pérdida de bienestar; (d) la cuarta red financiera está pendiente de calibración; (e) el ejemplo de coevolución no constituye RQGM. Ninguna salida es asesoramiento financiero.
